In [5]:
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent
DATA_PATH = ROOT / "data" / "processed" / "hotel_bookings_clean.csv"

df = pd.read_csv(DATA_PATH)
print(df.shape)


(86637, 42)


In [6]:
# Step 1: the value of each booking that was cancelled
# (revenue = price per night x nights; is_canceled is 1 or 0, so this keeps
#  the value for cancelled bookings and gives 0 for kept ones)
df["cancelled_value"] = df["revenue"] * df["is_canceled"]


In [7]:

# Step 2: overall numbers
total_value = df["revenue"].sum()
lost_value = df["cancelled_value"].sum()
print("Total booking value (EUR):", round(total_value))
print("Value of cancelled bookings (EUR):", round(lost_value))
print("Share of BOOKINGS cancelled (%):", round(df["is_canceled"].mean() * 100, 1))
print("Share of VALUE cancelled (%):", round(lost_value / total_value * 100, 1))


Total booking value (EUR): 34446255
Value of cancelled bookings (EUR): 11478643
Share of BOOKINGS cancelled (%): 27.7
Share of VALUE cancelled (%): 33.3


In [8]:

# Step 3: the same numbers for each hotel
by_hotel = df.groupby("hotel").agg(
    bookings=("is_canceled", "size"),               # how many bookings
    cancel_rate=("is_canceled", "mean"),            # share cancelled
    value_cancelled=("cancelled_value", "sum"))     # money in cancelled bookings
by_hotel["cancel_rate"] = (by_hotel["cancel_rate"] * 100).round(1)
by_hotel["value_cancelled"] = by_hotel["value_cancelled"].round()
print(by_hotel)

              bookings  cancel_rate  value_cancelled
hotel                                               
City Hotel       53042         30.2        6609056.0
Resort Hotel     33595         23.7        4869587.0


In [9]:
# Step 4: the same numbers for each booking channel
by_channel = df.groupby("market_segment").agg(
    bookings=("is_canceled", "size"),
    cancel_rate=("is_canceled", "mean"),
    value_cancelled=("cancelled_value", "sum"))
by_channel["cancel_rate"] = (by_channel["cancel_rate"] * 100).round(1)
by_channel["share_of_lost_value"] = (by_channel["value_cancelled"] / lost_value * 100).round(1)
by_channel["value_cancelled"] = by_channel["value_cancelled"].round()
print(by_channel.sort_values("value_cancelled", ascending=False))

                bookings  cancel_rate  value_cancelled  share_of_lost_value
market_segment                                                             
Online TA          51285         35.5        9292902.0                 81.0
Direct             11652         14.9         929714.0                  8.1
Offline TA/TO      13748         14.9         815609.0                  7.1
Groups              4891         27.2         315620.0                  2.7
Corporate           4155         12.2         110415.0                  1.0
Aviation             222         19.8          14065.0                  0.1
Complementary        682         12.2            270.0                  0.0
Undefined              2        100.0             48.0                  0.0


In [13]:
# ASSUMPTION: in each group, the cancellation rate falls by 5 percentage points
# (for example from 35.5% down to 30.5%). Change this number to test other cases.
reduction = 5

# A small function: give it a group of bookings and a name,
# and it works out how much value would be saved

def what_if(group, name):
    n = len(group)                                                   # bookings in the group
    current_rate = group["is_canceled"].mean() * 100                 # today's cancellation rate (%)
    avg_value = group[group["is_canceled"] == 1]["revenue"].mean()   # average value of a cancelled booking
    saved_bookings = n * reduction / 100                             # bookings that would now be kept
    saved_value = saved_bookings * avg_value                         # money those bookings are worth
    return {"Group": name,
            "Bookings": n,
            "Current rate %": round(current_rate, 1),
            "Rate after %": round(current_rate - reduction, 1),
            "Avg value of a cancelled booking": round(avg_value),
            "Bookings saved": round(saved_bookings),
            "Value saved": round(saved_value),
            "Share of all lost value %": round(saved_value / lost_value * 100, 1)}

In [14]:
# The three groups we want to test
online = df[df["market_segment"] == "Online TA"]
early = df[df["lead_time"] > 90]
city = df[df["hotel"] == "City Hotel"]

In [15]:
# Run the function for each group and show the results as one table
table = pd.DataFrame([what_if(online, "Online TA bookings"),
                      what_if(early, "Booked more than 90 days ahead"),
                      what_if(city, "City Hotel bookings")])
print(table.to_string(index=False))

                         Group  Bookings  Current rate %  Rate after %  Avg value of a cancelled booking  Bookings saved  Value saved  Share of all lost value %
            Online TA bookings     51285            35.5          30.5                               510            2564      1307149                       11.4
Booked more than 90 days ahead     29915            36.9          31.9                               545            1496       814939                        7.1
           City Hotel bookings     53042            30.2          25.2                               413            2652      1094056                        9.5


A 5-point reduction in cancellations among Online TA bookings would keep about 2,600 bookings, worth about €1.3M over the data period (about 11% of all cancelled value). The same improvement for bookings made more than 90 days ahead is worth about €0.8M, and for the City Hotel about €1.1M. These are scenario estimates, and the groups overlap, so they should not be added together.

# Recommendations

| Priority | Recommendation | Evidence from our analysis | Expected impact |
|---|---|---|---|
| **1** | **Target Online TA bookings first.** Ask the big online agents for stricter cancellation terms on early bookings, and add incentives to book direct. | H3: Online TA cancels 35.5% vs 16.3% for other channels (Z = 62.3, p < 0.001). It is 59% of bookings but **81% of cancelled value**. Direct bookings cancel only 14.9%. | A 5-point cut is worth about **€1.3M** (11% of lost value) |
| **2** | **Add rules for early bookings.** Send reminders and ask for re-confirmation, and shorten the free-cancellation window for bookings made far ahead. | H2: cancelled bookings were made 106 vs 71 days ahead (t = 52.2, p < 0.001). Cancellation rises from 8.5% (0-7 days) to about 40% (180+ days). | A 5-point cut for bookings >90 days ahead is worth about **€0.8M** |
| **3** | **Flag high-risk bookings for early contact.** | Step 3: guests with no special requests cancel 33.5% vs 16.2% for three or more. Guests with one previous cancellation cancel 76.4% (1,401 bookings). | Lets the team focus reminders where risk is highest |
| **4** | **Use different policies for the two hotels.** Tighter rules and reminders at the City Hotel, and protect what already works at the Resort. | H1: City 30.2% vs Resort 23.7% (Z = 20.8, p < 0.001). City holds €6.6M of cancelled value vs €4.9M. | A 5-point cut at City is worth about **€1.1M** |
| **5** | **Do NOT assume stricter deposits are the fix. Investigate first.** Check how non-refundable bookings are made and whether they are really charged, then pilot before rolling out. | H4: deposit type is related to cancellation (chi-square = 2354.6, p < 0.001), but Non Refund bookings cancel **94.7%** of the time. They are only 1.2% of bookings. | Avoids a policy that could backfire |
| **6** | **Match pricing and terms to the season.** Run winter offers for the weak months and protect peak-season rooms with firmer terms. | H5: average ADR is €138 in summer vs €77 in winter (F = 7436, p < 0.001). Season explains 20.5% of price variation. Bookings are lowest from November to January. | Higher value protected where an empty room costs most |

## How to measure success
Track these every month, by hotel and by channel:
- Cancellation rate (%)
- Value of cancelled bookings (EUR)
- Share of bookings coming from Online TA vs Direct

## Limitations
- **The results show association, not cause.** For example, the Non Refund result does not prove that deposits cause cancellations.
- **"Value" is potential revenue.** Cancelled rooms may be resold, and refunds or charges are not in the data.
- **What-if figures are scenarios.** The 5-point reduction is our assumption, and the groups overlap, so the amounts must not be added together.
- **The data covers July 2015 to August 2017** and 32,000 duplicate rows were removed. With no booking ID, some may have been genuine bookings.
- **Two columns were deliberately not used as predictors:** `reservation_status` and `room_changed`, because they are recorded after the cancellation happens (data leakage).